# MVFoul — Unified Pipeline

**Everything in one notebook.** This notebook combines dataset exploration (statistics and plots) with all three baseline models, so you never need to transfer files between notebooks.

| Part | What it does | Time |
|------|-------------|------|
| **Part 1** | Install packages | ~1 min |
| **Part 2** | Download SoccerNet-MVFoul data | ~2 min |
| **Part 3** | Dataset statistics and 7 plots | ~1 min |
| **Part 4** | Prepare data for training | ~1 min |
| **Part 5** | Approach A — MViT video baseline | ~5 min (debug) |
| **Part 6** | Approach B — Pose + BiLSTM | ~3 min (debug) |
| **Part 7** | Approach C — ST-GCN (novel) | ~3 min (debug) |
| **Part 8** | Compare all three approaches | ~1 min |
| **Part 9** | Download your results | manual |

**How to run:** Click **Runtime > Run all** or run each cell one at a time from top to bottom.

**GPU required for Parts 5–8.** Go to **Runtime > Change runtime type** and select **T4 GPU** before running.

---
## Part 1 — Install packages

This cell installs everything both the statistics section and the training section need. It only needs to run once per Colab session.

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install SoccerNet --quiet
!pip install matplotlib seaborn pandas numpy --quiet
!pip install pytorchvideo --quiet
!pip install mediapipe --quiet
!pip install ultralytics --quiet
!pip install av --quiet
!pip install scikit-learn --quiet

print('All packages installed.')

---
## Part 2 — Imports, config, and data download

### Cell 2a — Imports and settings

This cell loads all the Python libraries we need and sets up directory paths.

**`DEBUG_MODE`** controls how much data the training sections (Parts 5–8) use:
- `True` (default) — uses only 60 actions, runs in ~10 minutes total. Good for checking everything works.
- `False` — uses the full dataset. Takes much longer but gives real results.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json, os, zipfile, random, time, warnings
from pathlib import Path
from collections import Counter, defaultdict

# ── Data and plotting ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Deep learning (used in Parts 5–8) ─────────────────────────────────────────
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

# ── SoccerNet ──────────────────────────────────────────────────────────────────
import SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit these if needed
# ══════════════════════════════════════════════════════════════════════════════
DEBUG_MODE  = True       # True = fast test run (~10 min), False = full run
DEBUG_N     = 60         # number of actions to use in debug mode
SEED        = 42
DATA_DIR    = Path('/content/mvfoul_data')
MVFOUL_DIR  = DATA_DIR / 'mvfouls'
CACHE_DIR   = Path('/content/feature_cache')
CKPT_DIR    = Path('/content/checkpoints')
RESULTS_DIR = Path('/content/results')

for d in [DATA_DIR, CACHE_DIR, CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Plotting style ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

# ── Color palette — consistent across all plots ───────────────────────────────
DIVE_COLOR   = '#E24B4A'
OTHER_COLOR  = '#378ADD'
CARD_COLORS  = {
    'No card':     '#3B8BD4',
    'Yellow card': '#EF9F27',
    'Red card':    '#E24B4A',
    'No card/Yellow card': '#9FE1CB',
    'Yellow card/Red card': '#F0997B',
}

# ── Action class labels (used by training in Parts 5–8) ──────────────────────
ACTION_CLASSES = ['Tackling','Standing tackling','High leg','Holding',
                  'Pushing','Elbowing','Challenge','Dive']
CLASS2IDX = {c: i for i, c in enumerate(ACTION_CLASSES)}
N_CLASSES  = len(ACTION_CLASSES)
DIVE_IDX   = CLASS2IDX['Dive']
SEVERITY_MAP = {
    '1.0': 'No card',
    '3.0': 'Yellow card',
    '5.0': 'Red card',
    '2.0': 'No card/Yellow card',
    '4.0': 'Yellow card/Red card',
    '':    'Unknown',
}

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  VRAM: {mem:.1f} GB')
else:
    print('WARNING: No GPU detected. Parts 5-8 will be very slow.')
    print('Go to Runtime > Change runtime type > T4 GPU, then re-run.')

print(f'\nDEBUG_MODE = {DEBUG_MODE}')
print(f'SoccerNet version: {SoccerNet.__version__}')
print(f'Dive index: {DIVE_IDX}  N_CLASSES: {N_CLASSES}')
print('Imports complete.')

### Cell 2b — Download MVFoul annotations

This downloads the SoccerNet-MVFoul annotation files. The annotations are small (~1 MB each) and public — no password needed.

If you also want the video clips (needed for real training, not debug mode), register at [soccer-net.org](https://www.soccer-net.org/) and replace the password below.

In [ ]:
# ── Download settings ─────────────────────────────────────────────────────────
# Annotations are public — leave password as-is for JSON-only download.
# To also download video clips, register at soccer-net.org and set your password.
SOCCERNET_PASSWORD = 'password'   # <-- replace if you have a SoccerNet account

mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
mySoccerNetDownloader.password = SOCCERNET_PASSWORD

# Download the MVFoul annotation zips (small, ~1 MB each)
mySoccerNetDownloader.downloadDataTask(
    task='mvfouls',
    split=['train', 'valid', 'test', 'challenge'],
    password=SOCCERNET_PASSWORD,
    verbose=True
)

# ── Unzip all downloaded archives ─────────────────────────────────────────────
for zip_path in sorted(MVFOUL_DIR.glob('*.zip')):
    split_name = zip_path.stem.replace('_720p', '')
    out_dir = MVFOUL_DIR / split_name
    if out_dir.exists():
        print(f'  {zip_path.name} already extracted')
        continue
    print(f'  Extracting {zip_path.name} ...', end=' ')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    print('done')

# ── List what we have ─────────────────────────────────────────────────────────
print('\nFiles after extraction:')
for p in sorted(MVFOUL_DIR.rglob('*.json')):
    print(f'  {p.relative_to(DATA_DIR)}  ({p.stat().st_size / 1024:.1f} KB)')

### Cell 2c — Parse annotations into a DataFrame

This reads the JSON annotation files and creates a single table (`df`) with one row per action. This `df` variable is used by both the statistics section (Part 3) and the training section (Parts 4–8).

In [ ]:
# ── Annotation keys present in the MVFoul JSON ────────────────────────────────
ANNOTATION_KEYS = [
    'Offence', 'Severity', 'Action class', 'Bodypart', 'Contact',
    'handball', 'handball_offence', 'trytoplay', 'touchball', 'upper_body_part'
]


def load_split(json_path: Path, split_name: str) -> list:
    """Parse one MVFoul annotation JSON into a list of flat dicts."""
    with open(json_path) as f:
        raw = json.load(f)

    actions = raw.get('Actions', raw.get('Set', {}))
    if not actions:
        print(f'  WARNING: no Actions found in {json_path.name}. Keys: {list(raw.keys())}')
        return []

    rows = []
    for action_id, data in actions.items():
        clips = data.get('clips', [])
        severity_raw = str(data.get('Severity', '')).strip()
        row = {
            'action_id':      action_id,
            'split':          split_name,
            'n_clips':        len(clips),
            'has_replay':     any(
                'replay' in c.get('path', '').lower() for c in clips
            ),
            'severity_raw':   severity_raw,
            'Severity':       SEVERITY_MAP.get(severity_raw, severity_raw),
        }
        for k in ANNOTATION_KEYS:
            if k == 'Severity':
                continue
            row[k] = str(data.get(k, 'Unknown')).strip()
        rows.append(row)
    return rows


# ── Discover annotation JSONs ─────────────────────────────────────────────────
split_map = {}
for json_path in sorted(MVFOUL_DIR.rglob('*.json')):
    path_str = str(json_path).lower()
    for s in ['train', 'valid', 'test', 'challenge']:
        if s in path_str and s not in split_map:
            split_map[s] = json_path
            break

print('Splits found:', list(split_map.keys()))
if not split_map:
    print('\nNo JSON files found. Check that the download cell ran without errors.')
    print('Expected layout: DATA_DIR/mvfouls/<split>/<anything>.json')
    for p in sorted(MVFOUL_DIR.rglob('*')):
        print(f'  {p.relative_to(MVFOUL_DIR)}')

# ── Load all splits into one DataFrame ────────────────────────────────────────
all_rows = []
for split_name, json_path in split_map.items():
    rows = load_split(json_path, split_name)
    all_rows.extend(rows)
    print(f'  {split_name:12s} -> {len(rows):5d} actions  (from {json_path.name})')

if not all_rows:
    raise RuntimeError('No actions loaded. See warnings above.')

df = pd.DataFrame(all_rows)
df['is_dive'] = df['Action class'].str.lower().str.contains('dive', na=False)
df['Action class'] = df['Action class'].str.strip()

print(f'\nTotal actions loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

---
## Part 3 — Dataset statistics and plots

These cells explore the dataset before training anything. You will see:
- How many actions are in each split (train/valid/test)
- How rare Dive/Simulation is compared to other fouls
- Contact rates, severity, body part, and intent signals
- Class weights needed for training

### Cell 3a — Summary statistics

In [ ]:
print('=' * 60)
print('SOCCERNET-MVFOUL — DATASET SUMMARY')
print('=' * 60)

# ── Split counts ──────────────────────────────────────────────────────────────
print('\n── Actions per split ──')
print(df.groupby('split').size().rename('count').to_string())

# ── Action class distribution ─────────────────────────────────────────────────
print('\n── Action class distribution (all splits) ──')
class_counts = df['Action class'].value_counts()
class_pct    = (class_counts / len(df) * 100).round(2)
class_df     = pd.DataFrame({'count': class_counts, 'pct': class_pct})
class_df.columns = ['count', '%']
print(class_df.to_string())

# ── Dive/Simulation breakdown ─────────────────────────────────────────────────
n_dive     = df['is_dive'].sum()
n_total    = len(df)
n_non_dive = n_total - n_dive

print(f'\n── Dive / Simulation ──')
print(f'  Dive/Simulation actions : {n_dive:5d}  ({n_dive/n_total*100:.2f}% of dataset)')
print(f'  All other fouls         : {n_non_dive:5d}  ({n_non_dive/n_total*100:.2f}%)')
print(f'  Class imbalance ratio   : 1 : {n_non_dive/max(n_dive,1):.0f}')

# ── Dive per split ────────────────────────────────────────────────────────────
print('\n── Dive actions per split ──')
print(
    df.groupby('split')['is_dive']
      .agg(['sum', 'mean'])
      .rename(columns={'sum': 'dive_count', 'mean': 'dive_rate'})
      .assign(dive_rate=lambda x: (x['dive_rate'] * 100).round(2))
      .to_string()
)

### Cell 3b — Plot 1: Foul type distribution (dive highlighted)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

class_order = df['Action class'].value_counts().index.tolist()
counts      = df['Action class'].value_counts()[class_order]
colors      = [DIVE_COLOR if 'Dive' in c or 'Simulation' in c else OTHER_COLOR
               for c in class_order]

bars = ax.barh(class_order[::-1], counts[class_order[::-1]].values,
               color=colors[::-1], edgecolor='white', linewidth=0.5)

for bar, count in zip(bars, counts[class_order[::-1]].values):
    pct = count / len(df) * 100
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
            f'{count:,}  ({pct:.1f}%)',
            va='center', fontsize=10, color='#444')

ax.set_xlabel('Number of actions', labelpad=8)
ax.set_title('MVFoul — Action class distribution (Dive/Simulation highlighted)')
ax.set_xlim(0, counts.max() * 1.35)
ax.tick_params(axis='y', labelsize=11)

legend_patches = [
    mpatches.Patch(color=OTHER_COLOR, label='Other foul types'),
    mpatches.Patch(color=DIVE_COLOR,  label='Dive / Simulation'),
]
ax.legend(handles=legend_patches, loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig('/content/plot1_foul_distribution.png', bbox_inches='tight')
plt.show()

### Cell 3c — Plot 2: Severity distribution — Dive vs. everything else

In [ ]:
severity_order = ['No card', 'No card/Yellow card', 'Yellow card',
                  'Yellow card/Red card', 'Red card']

def severity_dist(sub_df):
    counts = sub_df['Severity'].value_counts()
    total  = len(sub_df)
    return {s: counts.get(s, 0) / total * 100 for s in severity_order}

dive_sev     = severity_dist(df[df['is_dive']])
nondive_sev  = severity_dist(df[~df['is_dive']])

x    = np.arange(len(severity_order))
w    = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - w/2, [nondive_sev[s] for s in severity_order],
            w, label='Other fouls', color=OTHER_COLOR, alpha=0.88)
b2 = ax.bar(x + w/2, [dive_sev[s]    for s in severity_order],
            w, label='Dive / Simulation', color=DIVE_COLOR, alpha=0.88)

for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        if h > 0.5:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(severity_order, rotation=20, ha='right')
ax.set_ylabel('% of subgroup')
ax.set_title('Severity distribution — Dive/Simulation vs. other fouls')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('/content/plot2_severity_comparison.png', bbox_inches='tight')
plt.show()

### Cell 3d — Plot 3: Contact analysis

The contact/no-contact split is the most direct signal for the contact-response hypothesis.

In [ ]:
contact_cross = (
    df.groupby(['is_dive', 'Contact'])
      .size()
      .rename('count')
      .reset_index()
)
contact_cross['label'] = contact_cross['is_dive'].map(
    {True: 'Dive/Simulation', False: 'Other fouls'}
)

totals = contact_cross.groupby('label')['count'].transform('sum')
contact_cross['pct'] = contact_cross['count'] / totals * 100

print('── Contact breakdown ──')
print(contact_cross.pivot(index='label', columns='Contact', values='pct').round(1).to_string())

pivot  = contact_cross.pivot(index='label', columns='Contact', values='pct').fillna(0)
colors = ['#5DCAA5', '#D85A30']

fig, ax = plt.subplots(figsize=(7, 4))
pivot.plot(kind='bar', ax=ax, color=colors, edgecolor='white', linewidth=0.5,
           rot=0, width=0.55)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

ax.set_xlabel('')
ax.set_ylabel('% of subgroup')
ax.set_title('Contact rate — Dive/Simulation vs. other fouls\n'
             '(key signal for contact-response hypothesis)')
ax.legend(title='Contact', frameon=False, bbox_to_anchor=(1, 1))
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig('/content/plot3_contact_analysis.png', bbox_inches='tight')
plt.show()

### Cell 3e — Plot 4: Body part and upper-body sub-type for dives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Body part (upper vs under) ────────────────────────────────────────────────
bp_cross = (
    df.groupby(['is_dive', 'Bodypart'])
      .size()
      .rename('count')
      .reset_index()
)
bp_cross['label'] = bp_cross['is_dive'].map({True: 'Dive', False: 'Other'})
bp_pivot = bp_cross.pivot(index='label', columns='Bodypart', values='count').fillna(0)
bp_pivot = bp_pivot.div(bp_pivot.sum(axis=1), axis=0) * 100

bp_pivot.plot(kind='bar', ax=axes[0], color=['#378ADD', '#BA7517'],
              edgecolor='white', rot=0, width=0.5)
for c in axes[0].containers:
    axes[0].bar_label(c, fmt='%.1f%%', padding=2, fontsize=9)
axes[0].set_ylim(0, 120)
axes[0].set_title('Body part involved')
axes[0].set_xlabel('')
axes[0].set_ylabel('% of subgroup')
axes[0].legend(title='Body part', frameon=False)

# ── Upper-body sub-type (only for dives with upper body) ──────────────────────
dive_upper = df[(df['is_dive']) & (df['Bodypart'] == 'Upper body')]
if len(dive_upper) > 0 and 'upper_body_part' in dive_upper.columns:
    ub_counts = dive_upper['upper_body_part'].value_counts()
    colors_ub = sns.color_palette('Blues_d', len(ub_counts))[::-1]
    ub_counts.plot(kind='barh', ax=axes[1], color=colors_ub, edgecolor='white')
    for i, v in enumerate(ub_counts):
        axes[1].text(v + 0.2, i, f'{v}', va='center', fontsize=9)
    axes[1].set_title('Upper-body sub-type\n(Dive/Simulation, upper body only)')
    axes[1].set_xlabel('Count')
else:
    axes[1].text(0.5, 0.5, 'No upper-body sub-type data\nfor Dive/Simulation',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11)
    axes[1].set_title('Upper-body sub-type (Dive only)')

plt.suptitle('Body part analysis — Dive/Simulation vs. other fouls', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot4_bodypart.png', bbox_inches='tight')
plt.show()

### Cell 3f — Plot 5: "Try to play" and "Touch ball" for dives

In [ ]:
intent_cols = ['trytoplay', 'touchball']
intent_labels = {'trytoplay': 'Tried to play ball', 'touchball': 'Touched ball'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, intent_cols):
    if col not in df.columns:
        ax.text(0.5, 0.5, f'Column "{col}" not in dataset',
                ha='center', va='center', transform=ax.transAxes)
        continue

    cross = (
        df.groupby(['is_dive', col])
          .size()
          .rename('count')
          .reset_index()
    )
    cross['label'] = cross['is_dive'].map({True: 'Dive', False: 'Other'})
    pivot = cross.pivot(index='label', columns=col, values='count').fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100

    palette = sns.color_palette('Set2', pivot.shape[1])
    pivot.plot(kind='bar', ax=ax, color=palette, edgecolor='white', rot=0, width=0.5)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', padding=2, fontsize=9)
    ax.set_ylim(0, 120)
    ax.set_title(intent_labels[col])
    ax.set_xlabel('')
    ax.set_ylabel('% of subgroup')
    ax.legend(frameon=False)

plt.suptitle('Intent signals — Dive/Simulation vs. other fouls\n'
             '(supports contact-response hypothesis)', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot5_intent_signals.png', bbox_inches='tight')
plt.show()

### Cell 3g — Plot 6: Full cross-tab heatmap

In [ ]:
cross_rows = []
feature_cols = ['Offence', 'Severity', 'Bodypart', 'Contact',
                'trytoplay', 'touchball', 'handball']

for col in feature_cols:
    if col not in df.columns:
        continue
    for val, sub in df.groupby(col):
        dive_rate = sub['is_dive'].mean() * 100
        cross_rows.append({
            'feature':    col,
            'value':      str(val),
            'n':          len(sub),
            'n_dive':     int(sub['is_dive'].sum()),
            'dive_rate':  round(dive_rate, 2)
        })

cross_df = pd.DataFrame(cross_rows)

pivot = cross_df.pivot(index='value', columns='feature', values='dive_rate').fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    pivot,
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Dive/Simulation rate (%)', 'shrink': 0.7},
    ax=ax
)
ax.set_title('Dive/Simulation rate (%) within each annotation category\n'
             '(higher = this value more often co-occurs with a dive)',
             pad=12)
ax.set_xlabel('Annotation feature')
ax.set_ylabel('Annotation value')
plt.tight_layout()
plt.savefig('/content/plot6_dive_crosstab_heatmap.png', bbox_inches='tight')
plt.show()

print('\n── Top co-occurring values with Dive/Simulation ──')
print(cross_df.sort_values('dive_rate', ascending=False).head(10).to_string(index=False))

### Cell 3h — Plot 7: Hardest confusable class pairs

In [ ]:
confusable_features = {
    'no_contact':  df['Contact'] == 'Without contact',
    'no_offence':  df['Offence'] == 'No offence',
    'no_ball':     df['touchball'].str.contains('No touch', na=False),
    'no_try':      df['trytoplay'].str.contains('No try', na=False),
}

confusion_rows = []
for foul_class, sub in df.groupby('Action class'):
    row = {'foul_class': foul_class, 'n': len(sub)}
    for feat, mask in confusable_features.items():
        row[feat + '_pct'] = (sub.index.isin(df[mask].index)).sum() / len(sub) * 100
    row['dive_like_score'] = np.mean([row[f + '_pct'] for f in confusable_features])
    confusion_rows.append(row)

conf_df = pd.DataFrame(confusion_rows).sort_values('dive_like_score', ascending=False)

print('── Dive-like annotation profile per foul class ──')
print('(higher score = more annotations that superficially resemble a dive)\n')
display_cols = ['foul_class', 'n', 'no_contact_pct', 'no_offence_pct',
                'no_ball_pct', 'no_try_pct', 'dive_like_score']
print(conf_df[display_cols].to_string(index=False, float_format='{:.1f}'.format))

fig, ax = plt.subplots(figsize=(9, 4))
palette = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR
           for c in conf_df['foul_class']]

ax.barh(conf_df['foul_class'][::-1], conf_df['dive_like_score'][::-1],
        color=palette[::-1], edgecolor='white')
ax.set_xlabel('Composite dive-like score (avg of 4 annotation features, %)')
ax.set_title('Which foul classes most resemble a dive in annotation space?\n'
             '(confusable pairs to watch during training)')
ax.axvline(conf_df[conf_df['foul_class'].str.contains('Dive', na=False)]
           ['dive_like_score'].mean(),
           color=DIVE_COLOR, linestyle='--', linewidth=1.2, alpha=0.6,
           label='Dive/Simulation mean')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('/content/plot7_confusable_classes.png', bbox_inches='tight')
plt.show()

### Cell 3i — Class weights for training loss

Computes inverse-frequency class weights. These are used automatically by the training cells in Parts 5–8.

In [ ]:
# Use only train split for weight calculation
stats_train_df = df[df['split'] == 'train'].copy()
class_counts_train = stats_train_df['Action class'].value_counts()

# Inverse-frequency weights (normalised so mean = 1)
inv_freq = 1.0 / class_counts_train
stats_weights  = inv_freq / inv_freq.mean()

print('── Inverse-frequency class weights (for weighted loss) ──')
print('These are used automatically by the training cells below.\n')
print('class_weights = {')
for cls, w in stats_weights.sort_values(ascending=False).items():
    flag = '  # <-- Dive: highest weight' if 'Dive' in cls else ''
    print(f'    "{cls}": {w:.4f},{flag}')
print('}')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR for c in class_counts_train.index]
class_counts_train.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=35)
axes[0].set_title('Training set: raw class counts')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', labelsize=9)

w_colors = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR for c in stats_weights.index]
stats_weights.sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color=w_colors, edgecolor='white', rot=35)
axes[1].set_title('Inverse-frequency training weights')
axes[1].set_ylabel('Weight (normalised, mean = 1)')
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='mean = 1')
axes[1].legend(frameon=False, fontsize=9)
axes[1].tick_params(axis='x', labelsize=9)

plt.suptitle('Class imbalance and recommended loss weights', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot8_class_weights.png', bbox_inches='tight')
plt.show()

### Cell 3j — Export statistics to CSV

Saves CSV files so you have a local copy of the statistics. These are also saved to `notebooks/outputs/` in the repo.

In [ ]:
df.to_csv('/content/mvfoul_all_actions.csv', index=False)
print(f'Saved all {len(df):,} actions -> /content/mvfoul_all_actions.csv')

df[df['is_dive']].to_csv('/content/mvfoul_dives_only.csv', index=False)
print(f'Saved {df["is_dive"].sum():,} dive actions -> /content/mvfoul_dives_only.csv')

stats_weights.rename('weight').to_csv('/content/class_weights.csv')
print('Saved class weights -> /content/class_weights.csv')

cross_df.to_csv('/content/dive_crosstab.csv', index=False)
print('Saved cross-tab -> /content/dive_crosstab.csv')

print('\n── File summary ──')
for f in ['/content/mvfoul_all_actions.csv', '/content/mvfoul_dives_only.csv',
          '/content/class_weights.csv', '/content/dive_crosstab.csv']:
    size_kb = Path(f).stat().st_size / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

### Cell 3k — Sanity check against the VARS paper

In [ ]:
paper_values = {
    'Standing tackling': 43.6,
    'Tackling':          15.6,
    'Challenge':         13.0,
    'Holding':           12.5,
    'Elbowing':           5.9,
    'High leg':           3.5,
    'Pushing':            2.9,
    'Dive':               0.9,
}

print('── Sanity check: our parse vs. VARS paper (Table 2) ──')
print(f'{"Action class":<26} {"Paper %":>10} {"Our %":>8} {"Diff":>8}')
print('-' * 56)
our_pcts = (df['Action class'].value_counts() / len(df) * 100)
for cls, paper_pct in sorted(paper_values.items(), key=lambda x: -x[1]):
    our_pct = our_pcts.get(cls, 0)
    diff    = our_pct - paper_pct
    flag    = '  <- check' if abs(diff) > 3 else ''
    print(f'{cls:<26} {paper_pct:>10.1f} {our_pct:>8.1f} {diff:>+8.1f}{flag}')

print('\n(Small differences expected if using full dataset vs. paper train split only.)')

### Cell 3l — Summary of key findings

In [ ]:
n_total    = len(df)
n_dive     = df['is_dive'].sum()
dive_pct   = n_dive / n_total * 100

dive_sev_mode = df[df['is_dive']]['Severity'].mode()[0] if n_dive > 0 else 'N/A'

dive_contact_pct  = (df[df['is_dive']]['Contact'] == 'With contact').mean() * 100
other_contact_pct = (df[~df['is_dive']]['Contact'] == 'With contact').mean() * 100

summary = f"""
DATASET SUMMARY — SoccerNet-MVFoul
===================================
Total actions      : {n_total:,}
Dive/Simulation    : {n_dive:,}  ({dive_pct:.2f}% — severely imbalanced)

Class imbalance ratio      : 1 dive per {n_total//max(n_dive,1)} other actions
Recommended training weight: ~{n_total/max(n_dive,1):.0f}x for Dive/Simulation

Most common severity for dives : {dive_sev_mode}
Contact rate — dives           : {dive_contact_pct:.1f}%
Contact rate — other fouls     : {other_contact_pct:.1f}%
-> Contact signal separability : {abs(dive_contact_pct - other_contact_pct):.1f} pp difference

Implication for the project:
  The contact-response hypothesis is supported — dives show a distinctly
  lower "With contact" rate than genuine fouls. However, the class imbalance
  is severe and weighted loss + balanced accuracy (not raw accuracy) must
  be used as the primary training and evaluation metric.
"""
print(summary)

---
## Part 4 — Prepare data for training

This section builds the train/valid DataFrames that the three model approaches need. It re-reads the same JSON files downloaded in Part 2 (no file upload needed) and creates a `train_df` and `valid_df` with the columns the models expect.

### Cell 4a — Build train/valid splits for training

In [ ]:
def load_split_df(json_path, split_name):
    """Load one split JSON into a DataFrame with columns needed for training."""
    with open(json_path) as f:
        raw = json.load(f)
    actions = raw.get('Actions', raw.get('Set', {}))
    rows = []
    for action_id, data in actions.items():
        clips  = data.get('clips', [])
        action = data.get('Action class', '').strip()
        if action not in CLASS2IDX:
            action = 'Dont know'
        rows.append({
            'action_id'  : action_id,
            'split'      : split_name,
            'label'      : CLASS2IDX.get(action, -1),
            'action_name': action,
            'is_dive'    : action == 'Dive',
            'clip_paths' : [c.get('path','') for c in clips],
            'Severity'   : SEVERITY_MAP.get(str(data.get('Severity','')).strip(), 'Unknown'),
        })
    return pd.DataFrame(rows)

split_dfs = {}
for json_path in sorted(MVFOUL_DIR.rglob('*.json')):
    p = str(json_path).lower()
    for s in ['train','valid','test']:
        if s in p and s not in split_dfs:
            split_dfs[s] = load_split_df(json_path, s)
            break

if not split_dfs:
    raise RuntimeError('No JSONs found. Make sure Part 2 ran successfully.')

for s in split_dfs:
    before = len(split_dfs[s])
    split_dfs[s] = split_dfs[s][split_dfs[s]['label'] >= 0].reset_index(drop=True)
    print(f'{s}: {len(split_dfs[s])} actions (dropped {before-len(split_dfs[s])} unknown)')

if DEBUG_MODE:
    for s in split_dfs:
        split_dfs[s] = (
            split_dfs[s]
            .groupby('label', group_keys=False)
            .apply(lambda x: x.sample(min(len(x), max(1, DEBUG_N // N_CLASSES)),
                                       random_state=SEED))
            .reset_index(drop=True)
        )
    print('DEBUG subsets:', {s: len(df) for s, df in split_dfs.items()})

train_df = split_dfs.get('train', pd.DataFrame())
valid_df  = split_dfs.get('valid', pd.DataFrame())
print(f'Train: {len(train_df)}  Valid: {len(valid_df)}')

### Cell 4b — Download diagnostic: do we have video clips?

In [ ]:
print('='*60)
print('DOWNLOAD DIAGNOSTIC')
print('='*60)

zip_files = sorted(MVFOUL_DIR.glob('*.zip'))
for z in zip_files:
    mb   = z.stat().st_size / 1e6
    kind = 'annotations only (<10MB)' if mb < 10 else 'includes video clips'
    print(f'  {z.name:30s} {mb:8.1f} MB  [{kind}]')

VIDEO_EXTS  = {'.mp4','.avi','.mkv','.mov'}
video_files = [p for p in MVFOUL_DIR.rglob('*') if p.suffix.lower() in VIDEO_EXTS]
print(f'JSON files : {len(list(MVFOUL_DIR.rglob("*.json")))}')
print(f'Video files: {len(video_files)}')

HAS_VIDEOS   = len(video_files) > 0
video_lookup = defaultdict(list)

if HAS_VIDEOS:
    for vp in video_files:
        video_lookup[vp.parent.name].append(vp)
    ann_ids = set(train_df['action_id']) | set(valid_df['action_id'])
    covered = ann_ids & set(video_lookup.keys())
    print(f'Coverage: {len(covered)}/{len(ann_ids)} annotated actions have clips')
    print('STATUS: VIDEO CLIPS FOUND')
else:
    print('STATUS: ANNOTATIONS ONLY -- SYNTHETIC MODE')
    print('  Models will train on random data (for pipeline validation only).')
    print('  To get real clips: register at soccer-net.org, set your password')
    print('  in Cell 2b, and re-run from Part 2.')

print(f'HAS_VIDEOS = {HAS_VIDEOS}')

### Cell 4c — Class weights and shared training utilities

In [ ]:
class_counts = train_df['label'].value_counts().sort_index()
counts_arr   = np.array([class_counts.get(i,1) for i in range(N_CLASSES)], dtype=float)
inv_freq     = 1.0 / counts_arr
weights      = torch.tensor(inv_freq / inv_freq.mean(), dtype=torch.float32).to(DEVICE)

print('Class weights (mean=1):')
for i, (cls, w) in enumerate(zip(ACTION_CLASSES, weights.cpu().numpy())):
    flag = '  <- Dive' if cls == 'Dive' else ''
    print(f'  [{i}] {cls:<22s} {w:.3f}{flag}')


def compute_metrics(y_true, y_pred):
    ba  = balanced_accuracy_score(y_true, y_pred)
    mac = f1_score(y_true, y_pred, average='macro',
                   labels=list(range(N_CLASSES)), zero_division=0)
    df1 = f1_score(y_true, y_pred, labels=[DIVE_IDX], average='macro', zero_division=0)
    cm  = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    return {'balanced_acc': round(ba,4), 'dive_f1': round(df1,4),
            'macro_f1': round(mac,4), 'conf_matrix': cm}


def plot_confusion_matrix(cm, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8,6))
    labels  = [c[:10] for c in ACTION_CLASSES]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.3, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)


def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss, preds, trues = 0.0, [], []
    for x, y in loader:
        x = x.to(DEVICE); y = y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
        preds.extend(logits.detach().argmax(1).cpu().numpy())
        trues.extend(y.cpu().numpy())
    return total_loss / len(trues), balanced_accuracy_score(trues, preds)


@torch.no_grad()
def run_eval(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        with autocast():
            preds.extend(model(x).argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    return compute_metrics(trues, preds)


def train_approach(name, model, train_dl, valid_dl,
                   n_epochs=15, lr=1e-4, patience=5, ckpt_name=None):
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler    = GradScaler()
    ckpt_path = CKPT_DIR / f'{ckpt_name or name}.pt'
    best_ba, patience_ctr = 0.0, 0
    history = []
    print(f'Training {name} | {n_epochs} epochs | lr={lr}')
    for ep in range(1, n_epochs+1):
        t0 = time.time()
        tr_loss, tr_ba = train_one_epoch(model, train_dl, optimizer, scaler, criterion)
        vm = run_eval(model, valid_dl)
        scheduler.step()
        history.append({'epoch':ep,'tr_loss':tr_loss,'tr_ba':tr_ba,
                        'val_ba':vm['balanced_acc'],'val_dive_f1':vm['dive_f1']})
        flag = ''
        if vm['balanced_acc'] > best_ba:
            best_ba = vm['balanced_acc']
            torch.save(model.state_dict(), ckpt_path)
            patience_ctr = 0; flag = ' [saved]'
        else:
            patience_ctr += 1
        elapsed = time.time() - t0
        print(f'  ep {ep:02d}/{n_epochs}  loss={tr_loss:.3f}  tr_ba={tr_ba:.3f}'
              f'  val_ba={vm["balanced_acc"]:.3f}  dive_f1={vm["dive_f1"]:.3f}'
              f'  ({elapsed:.1f}s){flag}')
        if patience_ctr >= patience:
            print(f'  Early stop at epoch {ep}'); break
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    final = run_eval(model, valid_dl)
    print(f'Best val_ba={best_ba:.4f}  dive_f1={final["dive_f1"]:.4f}')
    return final, pd.DataFrame(history)

print('Shared utilities ready.')

---
## Part 5 — Approach A: MViT Video Baseline

Replicates VARS (Held et al. 2023). Uses a pretrained MViTv2-S video model.

Two training phases:
1. **Frozen backbone** — only train the classification head (fast)
2. **Full fine-tune** — unfreeze all layers (slower, better results)

### Cell 5a — Video dataset and model definition

In [ ]:
def load_clip_frames(video_path, n_frames=16, size=224):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return None
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total-1,0), n_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok: frame = np.zeros((size,size,3), dtype=np.uint8)
        frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (size,size))
        frames.append(frame)
    cap.release()
    t    = torch.from_numpy(np.stack(frames)).float() / 255.0
    mean = torch.tensor([0.45,0.45,0.45]).view(1,1,1,3)
    std  = torch.tensor([0.225,0.225,0.225]).view(1,1,1,3)
    return ((t-mean)/std).permute(3,0,1,2)  # (3,T,H,W)


class MVFoulVideoDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=16, size=224):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.size=size
        self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            return torch.randn(3,self.n_frames,self.size,self.size), label
        clips   = self.vl.get(row['action_id'],[])
        replays = [c for c in clips if 'replay' in str(c).lower()]
        for vp in (replays or clips):
            t = load_clip_frames(vp, self.n_frames, self.size)
            if t is not None: return t, label
        return torch.randn(3,self.n_frames,self.size,self.size), label


class ApproachA_MViT(nn.Module):
    def __init__(self, n_classes, freeze_backbone=True):
        super().__init__()
        self.backbone = torch.hub.load(
            'facebookresearch/pytorchvideo', 'mvit_v2_s', pretrained=True
        )
        in_feat = self.backbone.head.proj.in_features
        self.backbone.head.proj = nn.Linear(in_feat, n_classes)
        if freeze_backbone:
            for nm, p in self.backbone.named_parameters():
                if 'head' not in nm: p.requires_grad = False
    def unfreeze(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def forward(self, x): return self.backbone(x)

print('Approach A defined.')

### Cell 5b — Train Approach A

In [ ]:
if not HAS_VIDEOS: print('SYNTHETIC MODE')

train_dl_A = DataLoader(MVFoulVideoDataset(train_df, video_lookup),
                        batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_A = DataLoader(MVFoulVideoDataset(valid_df, video_lookup),
                        batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

model_A = ApproachA_MViT(n_classes=N_CLASSES, freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in model_A.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_A.parameters())
print(f'Approach A | trainable: {trainable:,} / {total:,}')

# Phase 1: frozen backbone
metrics_A, hist_A1 = train_approach('A-frozen', model_A, train_dl_A, valid_dl_A,
    n_epochs=5, lr=3e-4, patience=3, ckpt_name='approach_A')

# Phase 2: full fine-tune
model_A.unfreeze()
metrics_A, hist_A2 = train_approach('A-full', model_A, train_dl_A, valid_dl_A,
    n_epochs=10, lr=5e-5, patience=5, ckpt_name='approach_A')

history_A = pd.concat([hist_A1, hist_A2], ignore_index=True)
print('Approach A final:', {k:v for k,v in metrics_A.items() if k!='conf_matrix'})

---
## Part 6 — Approach B: Pose + BiLSTM

Replicates Fang, Yeung & Fujii (2024). Extracts 33 body keypoints per frame using MediaPipe, then classifies the sequence with a bidirectional LSTM.

### Cell 6a — MediaPipe pose extraction

In [ ]:
import mediapipe as mp
mp_pose    = mp.solutions.pose
N_KP       = 33
N_FRAMES_B = 16


def extract_pose_sequence(video_path, n_frames=N_FRAMES_B):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return None
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total-1,0), n_frames, dtype=int)
    seq     = []
    with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                      min_detection_confidence=0.3) as pose:
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok: seq.append(np.zeros(N_KP*3)); continue
            result = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if result.pose_landmarks:
                kps = [[lm.x,lm.y,lm.visibility] for lm in result.pose_landmarks.landmark]
            else:
                kps = [[0.0,0.0,0.0]]*N_KP
            seq.append(np.array(kps).flatten())
    cap.release()
    return np.array(seq, dtype=np.float32)


def get_pose_cached(action_id, video_paths, n_frames=N_FRAMES_B):
    cache = CACHE_DIR / f'pose_{action_id}.npy'
    if cache.exists(): return np.load(cache)
    replays = [v for v in video_paths if 'replay' in str(v).lower()]
    for vp in (replays or video_paths):
        seq = extract_pose_sequence(vp, n_frames)
        if seq is not None:
            np.save(cache, seq); return seq
    seq = np.zeros((n_frames, N_KP*3), dtype=np.float32)
    np.save(cache, seq); return seq


if HAS_VIDEOS:
    all_ids = list(train_df['action_id']) + list(valid_df['action_id'])
    already = {f.stem.replace('pose_','') for f in CACHE_DIR.glob('pose_*.npy')}
    to_do   = [a for a in all_ids if a not in already]
    print(f'Pose cache: {len(already)} ready, {len(to_do)} to extract')
    for i, aid in enumerate(to_do):
        get_pose_cached(aid, video_lookup.get(aid,[]))
        if (i+1) % 20 == 0: print(f'  {i+1}/{len(to_do)}')
    print('Pose extraction complete')
else:
    print('Synthetic mode')

### Cell 6b — BiLSTM model and training

In [ ]:
class MVFoulPoseDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=N_FRAMES_B):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            return torch.randn(self.n_frames, N_KP*3), label
        clips = self.vl.get(row['action_id'],[])
        seq   = get_pose_cached(row['action_id'], clips, self.n_frames)
        return torch.from_numpy(seq), label


class ApproachB_BiLSTM(nn.Module):
    def __init__(self, in_features=N_KP*3, hidden=256, n_layers=2,
                 n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_features, hidden)
        self.lstm = nn.LSTM(hidden, hidden, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden*2, n_classes))
    def forward(self, x):
        x = F.relu(self.proj(x))
        out, _ = self.lstm(x)
        return self.head(out.mean(1))

train_dl_B = DataLoader(MVFoulPoseDataset(train_df, video_lookup),
                        batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_B = DataLoader(MVFoulPoseDataset(valid_df, video_lookup),
                        batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model_B = ApproachB_BiLSTM().to(DEVICE)
print(f'Approach B | params: {sum(p.numel() for p in model_B.parameters()):,}')

metrics_B, history_B = train_approach('B-BiLSTM', model_B, train_dl_B, valid_dl_B,
    n_epochs=30, lr=3e-4, patience=7, ckpt_name='approach_B')
print('Approach B final:', {k:v for k,v in metrics_B.items() if k!='conf_matrix'})

---
## Part 7 — Approach C: ST-GCN (Novel Contribution)

Velocity-augmented skeleton graphs. Node features: [x, y, vx, vy].
Pure skeleton approach — no video pixels needed.

### Cell 7a — Skeleton graph and feature extraction

In [ ]:
COCO_FROM_MP = [0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
N_JOINTS = 17
JOINT_NAMES = ['Nose','L-eye','R-eye','L-ear','R-ear',
               'L-shoulder','R-shoulder','L-elbow','R-elbow',
               'L-wrist','R-wrist','L-hip','R-hip',
               'L-knee','R-knee','L-ankle','R-ankle']
SKELETON_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]


def build_adjacency(n_joints, edges):
    A = np.zeros((n_joints,n_joints), dtype=np.float32)
    for i,j in edges: A[i,j]=A[j,i]=1.0
    A += np.eye(n_joints, dtype=np.float32)
    D = np.diag(A.sum(axis=1)**-0.5)
    return torch.tensor(D@A@D, dtype=torch.float32)

A_HAT = build_adjacency(N_JOINTS, SKELETON_EDGES)
print(f'A_HAT: {A_HAT.shape}  edges: {len(SKELETON_EDGES)}')


def pose_to_graph(seq):
    T    = seq.shape[0]
    full = seq.reshape(T, 33, 3)
    coco = full[:, COCO_FROM_MP, :2]  # (T,17,2)
    vel  = np.zeros_like(coco)
    vel[1:] = coco[1:] - coco[:-1]
    return np.concatenate([coco, vel], axis=-1).astype(np.float32)  # (T,17,4)


def get_graph_cached(action_id, video_paths, n_frames=N_FRAMES_B):
    cache = CACHE_DIR / f'graph_{action_id}.npy'
    if cache.exists(): return np.load(cache)
    pose_f = CACHE_DIR / f'pose_{action_id}.npy'
    if pose_f.exists():
        seq = np.load(pose_f)
    else:
        replays = [v for v in video_paths if 'replay' in str(v).lower()]
        seq = None
        for vp in (replays or video_paths):
            seq = extract_pose_sequence(vp, n_frames)
            if seq is not None: break
        if seq is None:
            seq = np.zeros((n_frames, 33*3), dtype=np.float32)
    feats = pose_to_graph(seq)
    np.save(cache, feats); return feats


if HAS_VIDEOS:
    all_ids = list(train_df['action_id']) + list(valid_df['action_id'])
    already = {f.stem.replace('graph_','') for f in CACHE_DIR.glob('graph_*.npy')}
    to_do   = [a for a in all_ids if a not in already]
    print(f'Graph cache: {len(already)} ready, {len(to_do)} to extract')
    for i, aid in enumerate(to_do):
        get_graph_cached(aid, video_lookup.get(aid,[]))
        if (i+1) % 50 == 0: print(f'  {i+1}/{len(to_do)}')
    print('Graph extraction complete')
else:
    print('Synthetic mode')

### Cell 7b — ST-GCN model

In [ ]:
class SpatialGCN(nn.Module):
    def __init__(self, c_in, c_out, A):
        super().__init__()
        self.register_buffer('A', A)
        self.W  = nn.Linear(c_in, c_out, bias=False)
        self.bn = nn.BatchNorm2d(c_out)
    def forward(self, x):
        B,C,T,N = x.shape
        h = x.permute(0,2,3,1).reshape(B*T,N,C)
        h = self.W(h)
        h = torch.bmm(self.A.unsqueeze(0).expand(B*T,-1,-1), h)
        h = h.reshape(B,T,N,-1).permute(0,3,1,2)
        return F.relu(self.bn(h))


class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, A, t_kernel=9, stride=1, dropout=0.2):
        super().__init__()
        self.gcn = SpatialGCN(c_in, c_out, A)
        self.tcn = nn.Sequential(
            nn.Conv2d(c_out, c_out, (t_kernel,1), (stride,1), (t_kernel//2,0)),
            nn.BatchNorm2d(c_out), nn.ReLU(), nn.Dropout2d(dropout))
        self.res = (nn.Sequential(nn.Conv2d(c_in,c_out,1,(stride,1)), nn.BatchNorm2d(c_out))
                    if c_in != c_out or stride != 1 else nn.Identity())
    def forward(self, x):
        return F.relu(self.tcn(self.gcn(x)) + self.res(x))


class ApproachC_STGCN(nn.Module):
    def __init__(self, c_in=4, n_classes=N_CLASSES, A=None, dropout=0.2):
        super().__init__()
        A = A if A is not None else A_HAT
        self.bn_in  = nn.BatchNorm1d(c_in*N_JOINTS)
        self.layers = nn.ModuleList([
            STGCNBlock(c_in, 64,  A, dropout=dropout),
            STGCNBlock(64,   128, A, dropout=dropout),
            STGCNBlock(128,  256, A, dropout=dropout),
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        B,C,T,N = x.shape
        xn = x.permute(0,2,1,3).reshape(B*T,C*N)
        xn = self.bn_in(xn).reshape(B,T,C,N).permute(0,2,1,3)
        for layer in self.layers: xn = layer(xn)
        return self.head(xn)

# Sanity check
_d = torch.randn(2,4,N_FRAMES_B,N_JOINTS).to(DEVICE)
_m = ApproachC_STGCN(A=A_HAT.to(DEVICE)).to(DEVICE)
_o = _m(_d)
print(f'ST-GCN: {tuple(_d.shape)} -> {tuple(_o.shape)}')
print(f'Approach C | params: {sum(p.numel() for p in _m.parameters()):,}')
del _d, _m, _o; torch.cuda.empty_cache()

### Cell 7c — ST-GCN training

In [ ]:
class MVFoulGraphDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=N_FRAMES_B):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            feats = np.random.randn(self.n_frames, N_JOINTS, 4).astype(np.float32)
        else:
            clips = self.vl.get(row['action_id'],[])
            feats = get_graph_cached(row['action_id'], clips, self.n_frames)
        return torch.from_numpy(feats).permute(2,0,1), label  # (4,T,N)

train_dl_C = DataLoader(MVFoulGraphDataset(train_df, video_lookup),
                        batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_C = DataLoader(MVFoulGraphDataset(valid_df, video_lookup),
                        batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model_C = ApproachC_STGCN(A=A_HAT.to(DEVICE)).to(DEVICE)

metrics_C, history_C = train_approach('C-STGCN', model_C, train_dl_C, valid_dl_C,
    n_epochs=40, lr=1e-3, patience=8, ckpt_name='approach_C')
print('Approach C final:', {k:v for k,v in metrics_C.items() if k!='conf_matrix'})

---
## Part 8 — Compare all three approaches

### Cell 8a — Results table

In [ ]:
all_results = {
    'A - MViT (video)'  : metrics_A,
    'B - BiLSTM (pose)' : metrics_B,
    'C - ST-GCN (novel)': metrics_C,
}
results_df = pd.DataFrame([
    {'Approach':name,'Balanced Acc':m['balanced_acc'],
     'Dive F1':m['dive_f1'],'Macro F1':m['macro_f1']}
    for name, m in all_results.items()
]).set_index('Approach')

print('='*65)
print('FINAL COMPARISON -- Validation Set')
print('='*65)
print(results_df.to_string(float_format='{:.4f}'.format))
if not HAS_VIDEOS:
    print('SYNTHETIC MODE -- numbers are random, re-run with real clips')

results_df.to_csv(RESULTS_DIR / 'comparison_table.csv')
print(f'Saved: {RESULTS_DIR}/comparison_table.csv')

### Cell 8b — Learning curves, confusion matrices, and bar chart

In [ ]:
# Learning curves
fig, axes = plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax, (name, hist, col) in zip(axes, [
    ('A MViT',  history_A, '#378ADD'),
    ('B BiLSTM',history_B, '#1D9E75'),
    ('C STGCN', history_C, '#E24B4A')]):
    ax.plot(hist['epoch'],hist['tr_ba'],color=col,alpha=0.4,linestyle='--',label='Train BA')
    ax.plot(hist['epoch'],hist['val_ba'],color=col,lw=2,label='Valid BA')
    ax.plot(hist['epoch'],hist['val_dive_f1'],color=col,lw=1.5,ls=':',label='Dive F1')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylim(0,1)
    ax.legend(fontsize=8,frameon=False)
axes[0].set_ylabel('Score')
plt.suptitle('Learning curves',y=1.02); plt.tight_layout()
plt.savefig(RESULTS_DIR/'learning_curves.png',bbox_inches='tight'); plt.show()

# Confusion matrices
fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax,(name,m) in zip(axes,all_results.items()):
    plot_confusion_matrix(m['conf_matrix'],name,ax)
plt.suptitle('Confusion matrices (row-normalised)',y=1.02); plt.tight_layout()
plt.savefig(RESULTS_DIR/'confusion_matrices.png',bbox_inches='tight'); plt.show()

# Bar comparison
fig, ax = plt.subplots(figsize=(9,4))
x, w = np.arange(len(results_df)), 0.25
b1=ax.bar(x-w,results_df['Balanced Acc'],w,label='Balanced Acc',color='#378ADD',alpha=0.85)
b2=ax.bar(x,  results_df['Dive F1'],     w,label='Dive F1',     color='#E24B4A',alpha=0.85)
b3=ax.bar(x+w,results_df['Macro F1'],    w,label='Macro F1',    color='#1D9E75',alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(results_df.index,fontsize=9)
ax.set_ylim(0,1.1); ax.set_ylabel('Score')
ax.set_title('Approach comparison -- validation set'); ax.legend(frameon=False)
for bars in (b1,b2,b3): ax.bar_label(bars,fmt='%.3f',padding=2,fontsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR/'bar_comparison.png',bbox_inches='tight'); plt.show()

### Cell 8c — ST-GCN joint importance ablation

In [ ]:
@torch.no_grad()
def joint_importance(model, valid_dl, joint_idx):
    model.eval()
    preds, trues = [], []
    for x, y in valid_dl:
        xc = x.clone().to(DEVICE)
        xc[:,:,:,joint_idx] = 0.0
        with autocast(): preds.extend(model(xc).argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    return f1_score(trues, preds, labels=[DIVE_IDX], average='macro', zero_division=0)

print('Joint importance ablation...')
baseline_f1 = metrics_C['dive_f1']
importance  = []
for j, jname in enumerate(JOINT_NAMES):
    drop = baseline_f1 - joint_importance(model_C, valid_dl_C, j)
    importance.append({'joint':jname,'dive_f1_drop':drop})
    print(f'  {jname:<14s} drop: {drop:+.4f}')

imp_df = pd.DataFrame(importance).sort_values('dive_f1_drop',ascending=False)
fig, ax = plt.subplots(figsize=(9,5))
colors  = ['#E24B4A' if d>0 else '#9FE1CB' for d in imp_df['dive_f1_drop']]
ax.barh(imp_df['joint'][::-1],imp_df['dive_f1_drop'][::-1],
        color=colors[::-1],edgecolor='white')
ax.axvline(0,color='gray',lw=0.8)
ax.set_xlabel('Dive F1 drop when joint zeroed')
ax.set_title('ST-GCN joint importance for Dive classification\n'
             '(contact-response: ankles/knees expected most diagnostic)')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'joint_importance.png',bbox_inches='tight'); plt.show()
imp_df.to_csv(RESULTS_DIR/'joint_importance.csv',index=False)
print('Top 3:', imp_df.head(3)[['joint','dive_f1_drop']].to_string(index=False))

### Cell 8d — Save all artefacts

In [ ]:
for nm in ['approach_A','approach_B','approach_C']:
    p = CKPT_DIR / f'{nm}.pt'
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'  {nm}.pt: {size}')

summary = {}
for nm, m in all_results.items():
    summary[nm] = {k: float(v) if not isinstance(v,np.ndarray) else v.tolist()
                  for k,v in m.items()}
with open(RESULTS_DIR/'full_results.json','w') as f:
    json.dump(summary, f, indent=2)

print(f'All outputs in {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1e3:.1f} KB)')

---
## Part 9 — Download your results

Colab sessions are **temporary** — files are deleted when the session ends. Download everything you need before closing this tab.

**Option A: Download individual files** using the file browser:
1. Click the **folder icon** in the left sidebar
2. Navigate to `/content/results/`
3. Click the **three dots** next to each file and select **Download**

**Option B: Run the cell below** to download the most important files automatically.

In [ ]:
from google.colab import files

print('Downloading result files...\n')

# Statistics CSVs
for f in ['/content/mvfoul_all_actions.csv', '/content/class_weights.csv',
          '/content/dive_crosstab.csv', '/content/mvfoul_dives_only.csv']:
    if Path(f).exists():
        files.download(f)
        print(f'  Downloaded: {f}')

# Training results
for f in sorted(RESULTS_DIR.iterdir()):
    files.download(str(f))
    print(f'  Downloaded: {f.name}')

print('\nDone! Check your browser Downloads folder.')
print('Copy the CSVs into notebooks/outputs/ in the repo.')